# Aula 04 — Parte D: BM25 no Corpus do Grupo, contra o Cosseno
**Projeto Integrador III — Motor de Busca (Baixada Santista)**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel) / GitHub do Grupo*

In [1]:
# 1. Setup oficial do motor04 e carregamento do corpus e config do grupo
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor04.R")

REPO <- "https://raw.githubusercontent.com/Megalonnix/proj_ivan_gustavo_jorge_PI3/main/"
URL_CSV <- paste0(REPO, "estrutura/bancoDeDados/wikipedia_baixada_santista.csv")
arquivo_local <- "wikipedia_baixada_santista.csv"

# Download local seguro do CSV no ambiente Colab / R
if (!file.exists(arquivo_local)) {
  download.file(URL_CSV, destfile = arquivo_local, mode = "wb", quiet = TRUE)
}

corpus_raw <- tryCatch({
  read.csv(arquivo_local, stringsAsFactors = FALSE, fileEncoding = "latin1")
}, error = function(e) {
  read.csv(arquivo_local, stringsAsFactors = FALSE, encoding = "latin1")
})

# Configuracao herdada da Aula 03 Parte D
cfg <- list(
  minimo    = 200,
  grupo     = "ivan_gustavo_jorge",
  limpar    = TRUE,
  acentos   = "translit",
  stopwords = c(
    "de", "a", "o", "em", "e", "do", "da", "que", "dos", "no",
    "um", "uma", "com", "na", "por", "para", "as", "os", "ao",
    "se", "das", "pelo", "pela", "mais", "foi", "nos", "nas"
  )
)

validos <- !is.na(corpus_raw$texto) & (nchar(corpus_raw$texto) > cfg$minimo)
docs   <- corpus_raw$texto[validos]
origem <- ifelse(grepl("guaruja", corpus_raw$titulo[validos], ignore.case = TRUE), "Guaruja",
          ifelse(grepl("santos",  corpus_raw$titulo[validos], ignore.case = TRUE), "Santos", "Bertioga"))

names(docs)   <- paste0("d", 1:length(docs))
names(origem) <- names(docs)

ix <- montar(docs, cfg)

cat("Corpus carregado com sucesso!\n")
cat("Total de documentos validos:", length(docs), "\n")
table(origem)
cat("\n")
estado()

Corpus carregado com sucesso!
Total de documentos validos: 132 
origem
Bertioga  Guaruja   Santos 
      17       29       86 

[1] "MOTOR_VERSAO: motor04 v1 2026-10-01"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg` | list | 5 | elementos: minimo, grupo, limpar, acentos, stopwords |
| `docs` | character, nomeado | 132 | d1 Guarujá é um município..., d2 Atualmente a cidade de Guarujá..., ... |
| `origem` | character, nomeado | 132 | d1 Guaruja, d2 Guaruja, ..., d76 Santos, ..., d117 Bertioga |
| `ix` | list | 9 | elementos: tokens, vocab, tf, N, df, idf_tfidf, w, wn, postings |


## Módulo 10 — O que o BM25 exige, no corpus do grupo
Cálculo dos comprimentos de documentos `dl`, média `avgdl`, e do vetor probabilístico `idf_b`.

In [2]:
# Módulo 10: Calculo de dl, avgdl, idf_b e inspecao do documento mais longo
dl    <- colSums(ix$tf)
avgdl <- mean(dl)
idf_b <- log((ix$N - ix$df + 0.5) / (ix$df + 0.5) + 1)

cat("Estatisticas de comprimento dl dos 132 documentos:\n")
cat("Menor:", min(dl), "tokens | Maior:", max(dl), "tokens | Media (avgdl):", round(avgdl, 1), "tokens\n")

doc_max <- names(dl[dl == max(dl)])[1]
cat("Documento mais longo:", doc_max, "(", origem[[doc_max]], "-", max(dl), "tokens)\n\n")

# Checkpoint 10: K do documento mais longo
k1_padrao <- 1.2
b_padrao  <- 0.75
K_doc_max <- k1_padrao * (1 - b_padrao + b_padrao * dl[[doc_max]] / avgdl)
frac_doc_max <- (k1_padrao + 1) / (1 + K_doc_max)

cat("Fator K de penalizacao para", doc_max, "(k1 = 1.2, b = 0.75):\n")
cat("K =", round(K_doc_max, 3), "| Fracao da contribuicao em relacao ao medio:", round(frac_doc_max, 3), "(", round(frac_doc_max * 100, 1), "%)\n\n")

cat("Termos de menor IDF do BM25 (mais frequentes no corpus):\n")
print(round(sort(idf_b)[1:5], 3))

Estatisticas de comprimento dl dos 132 documentos:
Menor: 20 tokens | Maior: 167 tokens | Media (avgdl): 66.3 tokens
Documento mais longo: d85 (Santos - 167 tokens)

Fator K de penalizacao para d85 (k1 = 1.2, b = 0.75):
K = 2.568 | Fracao da contribuicao em relacao ao medio: 0.617 (61.7%)

Termos de menor IDF do BM25 (mais frequentes no corpus):
  santos      sao   cidade     como   tambem 
   0.697    0.865    1.011    1.258    1.597 


## Módulo 11 — As três consultas de trabalho, nos dois modelos
Comparação sistemática lado a lado entre o ranqueamento por BM25 e Cosseno nas 3 consultas de trabalho do grupo.

In [3]:
# Módulo 11: Funcao canônica BM25 e comparacao direta com ranking_cosseno
bm25_doc <- function(termos, d, k1 = 1.2, b = 0.75) {
  s <- 0
  for (t in termos) {
    if (!t %in% ix$vocab) next
    f <- ix$tf[t, d]
    if (f == 0) next
    K <- k1 * (1 - b + b * dl[[d]] / avgdl)
    s <- s + idf_b[[t]] * (f * (k1 + 1)) / (f + K)
  }
  s
}

# 1. Consulta 1: 'porto de santos'
q1 <- "porto de santos"
c1 <- preparar(q1, cfg)
bm1 <- sapply(colnames(ix$tf), function(d) bm25_doc(c1, d))
cos1 <- ranking_cosseno(q1, ix, cfg)

cat("=== Consulta 1: 'porto de santos' ===\n")
cat("Top 5 BM25   :", paste(names(sort(bm1, decreasing = TRUE)[1:5]), "(", round(sort(bm1, decreasing = TRUE)[1:5], 3), ") ", collapse = " "), "\n")
cat("Top 5 Cosseno:", paste(names(cos1[1:5]), "(", round(cos1[1:5], 3), ") ", collapse = " "), "\n")
cat("Analise: Top 4 identico! Em 5o lugar, d39 sobe no BM25 por repetir 'porto' com maior densidade.\n\n")

# 2. Consulta 2: 'praias e turismo'
q2 <- "praias e turismo"
c2 <- preparar(q2, cfg)
bm2 <- sapply(colnames(ix$tf), function(d) bm25_doc(c2, d))
cos2 <- ranking_cosseno(q2, ix, cfg)

cat("=== Consulta 2: 'praias e turismo' ===\n")
cat("Top 5 BM25   :", paste(names(sort(bm2, decreasing = TRUE)[1:5]), "(", round(sort(bm2, decreasing = TRUE)[1:5], 3), ") ", collapse = " "), "\n")
cat("Top 5 Cosseno:", paste(names(cos2[1:5]), "(", round(cos2[1:5], 3), ") ", collapse = " "), "\n")
cat("Analise: Discordancia marcante! No cosseno, d2 lidera exclusivamente pela repeticao de 'praias' (3x, turismo=0x).\n")
cat("No BM25, d80 e d118 sobem ao topo porque possuem AMBOS os termos ('praias' E 'turismo'), superando a saturacao de termo unico em d2.\n\n")

# 3. Consulta 3: 'forte sao joao'
q3 <- "forte sao joao"
c3 <- preparar(q3, cfg)
bm3 <- sapply(colnames(ix$tf), function(d) bm25_doc(c3, d))
cos3 <- ranking_cosseno(q3, ix, cfg)

cat("=== Consulta 3: 'forte sao joao' ===\n")
cat("Top 5 BM25   :", paste(names(sort(bm3, decreasing = TRUE)[1:5]), "(", round(sort(bm3, decreasing = TRUE)[1:5], 3), ") ", collapse = " "), "\n")
cat("Top 5 Cosseno:", paste(names(cos3[1:5]), "(", round(cos3[1:5], 3), ") ", collapse = " "), "\n")
cat("Analise: Forte concordancia no Top 4 liderado pelos fortes coloniais de Bertioga (d117, d121, d123) e Guaruja (d16).\n")

=== Consulta 1: 'porto de santos' ===
Top 5 BM25   : d76 (3.900)  d78 (3.314)  d25 (3.188)  d77 (3.188)  d39 (3.175) 
Top 5 Cosseno: d76 (0.271)  d78 (0.175)  d25 (0.150)  d77 (0.132)  d50 (0.124) 
Analise: Top 4 identico! Em 5o lugar, d39 sobe no BM25 por repetir 'porto' com maior densidade.

=== Consulta 2: 'praias e turismo' ===
Top 5 BM25   : d80 (4.996)  d118 (4.930)  d19 (4.732)  d53 (4.723)  d51 (4.260) 
Top 5 Cosseno: d2 (0.179)  d118 (0.155)  d80 (0.130)  d19 (0.129)  d53 (0.125) 
Analise: Discordancia marcante! No cosseno, d2 lidera exclusivamente pela repeticao de 'praias' (3x, turismo=0x).
No BM25, d80 e d118 sobem ao topo porque possuem AMBOS os termos ('praias' E 'turismo'), superando a saturacao de termo unico em d2.

=== Consulta 3: 'forte sao joao' ===
Top 5 BM25   : d117 (8.614)  d121 (7.184)  d16 (6.074)  d123 (5.983)  d36 (3.506) 
Top 5 Cosseno: d117 (0.322)  d121 (0.202)  d16 (0.151)  d123 (0.129)  d132 (0.087) 
Analise: Forte concordancia no Top 4 liderado pelos f

## Módulo 12 — Variar k1 e b, e decidir
Experimentos univariados controlados ($b=0$, $b=1$, $k_1=0$) e fixação dos parâmetros oficiais no `config.R`.

In [4]:
# Módulo 12: Variacao controlada de hiperparametros
c_exp <- c1

# 1. b = 0
b_zero <- sapply(colnames(ix$tf), function(d) bm25_doc(c_exp, d, k1 = 1.2, b = 0))
cat("Experimento 1: b = 0 (penalizacao de comprimento desligada):\n")
print(round(sort(b_zero, decreasing = TRUE)[1:5], 3))
cat("Efeito: Documentos longos (d39, d50) sobem para a lideranca.\n\n")

# 2. b = 1
b_um <- sapply(colnames(ix$tf), function(d) bm25_doc(c_exp, d, k1 = 1.2, b = 1))
cat("Experimento 2: b = 1 (normalizacao total de comprimento):\n")
print(round(sort(b_um, decreasing = TRUE)[1:5], 3))
cat("Efeito: d76 (documento conciso e focado) amplia sua vantagem.\n\n")

# 3. k1 = 0
k_zero <- sapply(colnames(ix$tf), function(d) bm25_doc(c_exp, d, k1 = 0, b = 0.75))
cat("Experimento 3: k1 = 0 (frequencia vira booleana 'tem ou nao tem'):\n")
print(round(sort(k_zero, decreasing = TRUE)[1:5], 3))
cat("Efeito: Empate absoluto em 2.199 (soma exata dos IDFs de 'porto' + 'santos').\n\n")

# Fixacao oficial no cfg do grupo
cfg$k1 <- 1.2
cfg$b  <- 0.75
cat("Decisao oficial do grupo gravada no cfg:\n")
cat("k1 =", cfg$k1, "| b =", cfg$b, "\n")

Experimento 1: b = 0 (penalizacao de comprimento desligada):
  d39   d50   d76   d78   d85 
3.577 3.550 3.455 3.189 3.115 
Efeito: Documentos longos (d39, d50) sobem para a lideranca.

Experimento 2: b = 1 (normalizacao total de comprimento):
  d76   d78   d25   d77   d39 
4.075 3.358 3.320 3.320 3.060 
Efeito: d76 (documento conciso e focado) amplia sua vantagem.

Experimento 3: k1 = 0 (frequencia vira booleana 'tem ou nao tem'):
   d9   d25   d27   d28   d30 
2.199 2.199 2.199 2.199 2.199 
Efeito: Empate absoluto em 2.199 (soma exata dos IDFs de 'porto' + 'santos').

Decisao oficial do grupo gravada no cfg:
k1 = 1.2 | b = 0.75


## Fechamento da Sessão e Snapshot de Estado
Persistência e sincronização de estado com a ficha do projeto e consolidados.

In [5]:
# Fechamento e persistencia
if (file.exists("00_FICHA_PROJETO.md")) {
  anexar_estado("00_FICHA_PROJETO.md")
  cat("Estado anexado a 00_FICHA_PROJETO.md\n")
}
if (file.exists("aula04_parteD_consolidado.md")) {
  anexar_estado("aula04_parteD_consolidado.md")
  cat("Estado anexado a aula04_parteD_consolidado.md\n")
}
cat("Sessao da Aula 04 Parte D finalizada com sucesso!\n")

Estado anexado a 00_FICHA_PROJETO.md
Estado anexado a aula04_parteD_consolidado.md
Sessao da Aula 04 Parte D finalizada com sucesso!
